# Análisis de Sentimiento

## 1. Librerías y configuraciones

In [4]:
import os
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import Dataset
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
# Reproducibilidad
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Paths
OUTPUT_DIR = Path("2_Modelos/sentiment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_CSV = "1_Data/processed/sentiment_data.csv"  

# Device (MPS aware)
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("DEVICE:", DEVICE)

DEVICE: mps


## 2. Hiperparámetros

In [6]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
BATCH_SIZE = 16
NUM_EPOCHS = 5
LEARNING_RATE = 3e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
EARLY_STOPPING_PATIENCE = 3
EVAL_STEPS = 200
SAVE_STEPS = 200
LOGGING_STEPS = 50

LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}


## 3. Carga y preparación de datos

In [7]:
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")
import re

stop_words = set(stopwords.words("english"))

def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)  # solo letras y espacios
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/arielamishaancohen/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
df = pd.read_csv(DATA_CSV)
df = df.rename(columns={"full_content": "text"})
df.head()

,title,description,content,text,source,author,publishedAt,url,sentiment
0,Brockton mayoral race heats up as Derenoncourt...,Did a finalist to be Brockton's next mayor duc...,BROCKTON Did a finalist to be Brockton's next ...,BROCKTON — Did a finalist to be Brockton's nex...,Enterprise News,"Chris Helms, The Enterprise",2025-10-23T18:06:11Z,https://www.enterprisenews.com/story/news/poli...,neutral
1,Bruce Springsteen lent helping hand to directo...,"Bruce Springsteen, the legendary musician, dem...",<ul><li>News</li>\r\n<li>Bruce Springsteen len...,"Bruce Springsteen, the legendary musician, dem...",The Times of India,IANS,2025-10-23T18:05:51Z,https://timesofindia.indiatimes.com/entertainm...,neutral
2,Binance Founder Changpeng Zhao Receives Pardon...,U.S. President Donald Trump has granted Binanc...,U.S. President Donald Trump has granted Binanc...,Advertisement\n\nU.S. President Donald Trump h...,ZyCrypto,Brenda Ngari,2025-10-23T18:03:17Z,https://zycrypto.com/binance-founder-changpeng...,neutral
3,A Mandatory Paper Bag Fee May Be Coming to Phi...,Check phillymag.com each morning Monday throug...,"NewsPlus, Philly mob documentary makes a big s...",A Mandatory Paper Bag Fee May Be Coming to Phi...,phillymag.com,Victor Fiorillo,2025-10-23T18:03:12Z,https://www.phillymag.com/news/2025/10/23/plas...,neutral
4,Wilco and Billy Bragg to Perform Mermaid Avenu...,"""Mermaid Avenue Live"" is set to take place at ...",Wilco and Billy Bragg have announced they will...,Wilco and Billy Bragg have announced they will...,Consequence.net,Eddie Fu,2025-10-23T18:02:51Z,https://consequence.net/2025/10/wilco-billy-br...,neutral


In [13]:
df['text'] = df['text'].apply(clean_text)

### Mapear labels

In [14]:
if "sentiment" in df.columns:
    df["label"] = df["sentiment"].map(LABEL2ID)
else:
    # suponer que ya hay columna 'label' con ids
    df["label"] = df["label"].astype(int)

df.head()

,title,description,content,text,source,author,publishedAt,url,sentiment,label
0,Brockton mayoral race heats up as Derenoncourt...,Did a finalist to be Brockton's next mayor duc...,BROCKTON Did a finalist to be Brockton's next ...,brockton finalist brockton next mayor duck deb...,Enterprise News,"Chris Helms, The Enterprise",2025-10-23T18:06:11Z,https://www.enterprisenews.com/story/news/poli...,neutral,1
1,Bruce Springsteen lent helping hand to directo...,"Bruce Springsteen, the legendary musician, dem...",<ul><li>News</li>\r\n<li>Bruce Springsteen len...,bruce springsteen legendary musician demonstra...,The Times of India,IANS,2025-10-23T18:05:51Z,https://timesofindia.indiatimes.com/entertainm...,neutral,1
2,Binance Founder Changpeng Zhao Receives Pardon...,U.S. President Donald Trump has granted Binanc...,U.S. President Donald Trump has granted Binanc...,advertisement u president donald trump granted...,ZyCrypto,Brenda Ngari,2025-10-23T18:03:17Z,https://zycrypto.com/binance-founder-changpeng...,neutral,1
3,A Mandatory Paper Bag Fee May Be Coming to Phi...,Check phillymag.com each morning Monday throug...,"NewsPlus, Philly mob documentary makes a big s...",mandatory paper bag fee may coming philly stor...,phillymag.com,Victor Fiorillo,2025-10-23T18:03:12Z,https://www.phillymag.com/news/2025/10/23/plas...,neutral,1
4,Wilco and Billy Bragg to Perform Mermaid Avenu...,"""Mermaid Avenue Live"" is set to take place at ...",Wilco and Billy Bragg have announced they will...,wilco billy bragg announced perform collaborat...,Consequence.net,Eddie Fu,2025-10-23T18:02:51Z,https://consequence.net/2025/10/wilco-billy-br...,neutral,1


In [15]:

print("Distribución:")
print(df["sentiment"].value_counts())
print(df["sentiment"].value_counts(normalize=True))


Distribución:
sentiment
neutral     451
positive     96
negative     64
Name: count, dtype: int64
sentiment
neutral     0.738134
positive    0.157119
negative    0.104746
Name: proportion, dtype: float64


### Split train y val

In [17]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df["label"])

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")


Train: 488, Val: 61, Test: 62


## 4. Pesos de clase

In [18]:
labels = train_df["label"].values
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", class_weights)

Class weights: tensor([3.1895, 0.4519, 2.1126])


## 5. Tokenizer y datasets

In [19]:
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

# Función simple de tokenización para map
def preprocess_texts(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
    )


### Crear datasets de HuggingFace
(Cada uno con columnas 'input_ids', 'attention_mask', 'label')

In [20]:
train_ds = Dataset.from_pandas(train_df[["text", "label"]].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[["text", "label"]].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[["text", "label"]].reset_index(drop=True))

train_ds = train_ds.map(preprocess_texts, batched=True, remove_columns=["text"])
val_ds = val_ds.map(preprocess_texts, batched=True, remove_columns=["text"])
test_ds = test_ds.map(preprocess_texts, batched=True, remove_columns=["text"])

# Asegurar formato
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


Map:   0%|          | 0/488 [00:00<?, ? examples/s]

Map:   0%|          | 0/61 [00:00<?, ? examples/s]

Map:   0%|          | 0/62 [00:00<?, ? examples/s]

## 6. Cargar modelo

In [21]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(DEVICE)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


## 7. Trainer personalizado (weighted loss)

In [22]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights_tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Guardar pesos y forzarlos al device del modelo cuando se use
        self._class_weights = class_weights_tensor

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        device = model.device

        # Mover tensores de inputs al device
        inputs = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in inputs.items()}

        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Asegurarse que los pesos estén en el mismo device
        class_weights = self._class_weights.to(device)

        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

## 8. Métricas

In [23]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    f1_weight = f1_score(labels, preds, average='weighted')

    metrics = {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weight,
    }

    # Añadir f1 por clase
    f1s = f1_score(labels, preds, average=None)
    for i, val in enumerate(f1s):
        metrics[f"f1_{ID2LABEL[i]}"] = float(val)

    return metrics

## 9. Training Arguments y Entrenamiento

In [24]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="steps",
    #do_eval = True,
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    max_grad_norm=GRAD_CLIP,
    lr_scheduler_type="cosine",
)

trainer = WeightedTrainer(
    class_weights_tensor=class_weights,
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

In [25]:
train_result = trainer.train()
trainer.save_model(str(OUTPUT_DIR/"final_model"))

/Users/arielamishaancohen/Library/Python/3.10/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


## 10. Evaluar en test

In [26]:
model = DistilBertForSequenceClassification.from_pretrained(str(OUTPUT_DIR/"final_model"))
model.to(DEVICE)
trainer.model = model

In [28]:
predictions = trainer.predict(test_ds)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

/Users/arielamishaancohen/Library/Python/3.10/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [29]:
# Imprimir métricas y classification report
metrics = predictions.metrics
print(metrics)
print(classification_report(labels, preds, target_names=list(LABEL2ID.keys())))

{'test_loss': 1.0315327644348145, 'test_accuracy': 0.5483870967741935, 'test_f1_macro': 0.4189031257304349, 'test_f1_weighted': 0.576022471863902, 'test_f1_negative': 0.2857142857142857, 'test_f1_neutral': 0.6746987951807228, 'test_f1_positive': 0.2962962962962963, 'test_runtime': 0.6167, 'test_samples_per_second': 100.531, 'test_steps_per_second': 6.486}
              precision    recall  f1-score   support

    negative       0.25      0.33      0.29         6
     neutral       0.76      0.61      0.67        46
    positive       0.24      0.40      0.30        10

    accuracy                           0.55        62
   macro avg       0.41      0.45      0.42        62
weighted avg       0.62      0.55      0.58        62



## 11. Matriz de Confusión y Análisis de Errores

In [30]:
def plot_confusion_matrix(true_labels, pred_labels, out_path=OUTPUT_DIR/"confusion_matrix.png"):
    cm = confusion_matrix(true_labels, pred_labels)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=list(LABEL2ID.keys()), yticklabels=list(LABEL2ID.keys()))
    plt.xlabel('Predicción')
    plt.ylabel('Real')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    print('Confusion matrix guardada en', out_path)



In [31]:
def error_analysis_df(test_df_local, true_labels, pred_labels, predictions_probs):
    # test_df_local debe corresponder al test_df original alineado con test_ds
    rows = []
    for i, (t, p, probs) in enumerate(zip(true_labels, pred_labels, predictions_probs)):
        if t != p:
            rows.append({
                'text': test_df_local.iloc[i]['text'][:300],
                'true_label': ID2LABEL[int(t)],
                'pred_label': ID2LABEL[int(p)],
                'pred_confidence': float(np.max(probs)),
                'pred_probs': probs.tolist(),
            })
    return pd.DataFrame(rows)

## 12. Predicción Rápida

In [32]:
def predict_text(text):
    inputs = tokenizer(text, truncation=True, padding='max_length', max_length=MAX_LENGTH, return_tensors='pt')
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)[0].cpu().numpy()
    pred = int(np.argmax(probs))
    return {"text": text, "sentiment": ID2LABEL[pred], "confidence": float(probs[pred]), "probs": probs}


In [33]:
print(predict_text("Company stocks soar to record highs after strong earnings report"))

{'text': 'Company stocks soar to record highs after strong earnings report', 'sentiment': 'neutral', 'confidence': 0.4311980605125427, 'probs': array([0.2541074 , 0.43119806, 0.31469464], dtype=float32)}


## 13. Guardar Resultados

In [34]:
with open(OUTPUT_DIR/"training_metrics.json", "w") as f:
    json.dump(train_result.metrics, f, indent=2)


In [35]:
predictions = trainer.predict(test_ds)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids
y_test_df = test_df.reset_index(drop=True)

/Users/arielamishaancohen/Library/Python/3.10/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [36]:
# 1️⃣ Preparar dataframe con resultados
import pandas as pd
import numpy as np

pred_probs = torch.softmax(torch.tensor(predictions.predictions), dim=1).numpy()
pred_labels = np.argmax(predictions.predictions, axis=1)

results_df = y_test_df.copy()
results_df["true_label"] = [ID2LABEL[i] for i in labels]
results_df["pred_label"] = [ID2LABEL[i] for i in pred_labels]

# Agregar las probabilidades de cada clase
for i, label_name in ID2LABEL.items():
    results_df[f"prob_{label_name}"] = pred_probs[:, i]

# 2️⃣ Guardar a CSV o Parquet
results_df.to_csv("2_Modelos/sentiment/test_predictions.csv", index=False)
# o Parquet para mayor eficiencia
results_df.to_parquet("2_Modelos/sentiment/test_predictions.parquet", index=False)